# Data cleaning for the Index of Multiple Deprivation dataset

In [ ]:
1 hour 10 mins morning 11th June

## Data sources

The Index of Multiple Deprivation (IMD) dataset was published by the Ministry of Housing, Communities & Local Government (MHCLG) and was constructed by Oxford Consultants for Social Inclusion (OCSI) and Deprivation.org. This dataset was downloaded from the GOV.UK website, as an Excel file (.xlsx), using the link 'https://www.gov.uk/government/statistics/english-indices-of-deprivation-2025' on 9th June 2026. The official statistics on the source website were published on 30 October 2025 and last updated on 17 November 2025.


An additional dataset has also been utilised, to support the preprocessing of the IMD dataset, the necessity of which is detailed fully later in this notebook. In brief, the dataset is required for standardising the location data of the IMD dataset, to allow comparison to other datasets (such as the ‘Police recorded crime open data tables, year ending March 2013 onwards’ dataset) and improve overall ease of understanding.

The 'Local Authority District (December 2024) to LAU1 to ITL3 to ITL2 to ITL1 (January 2025) Lookup in the UK' dataset was published by the Office for National Statistics (ONS). The dataset was downloaded from the National Data Library website, as a csv file (.csv), using the link 'https://www.data.gov.uk/dataset/2b47adcc-62b6-4dd3-a6cb-271b3035e9fd/local-authority-district-december-2024-to-lau1-to-itl3-to-itl2-to-itl1-january-2025-lookup-in-t' on 9th June 2026. The dataset was last updated on 09 December 2025 and the csv file was added to the website on 12 January 2025. 


 - merge datasets on the LADs column (so that the columns are then LSOA, LAD, IOD IOD, ITL1, ITL2)
 - dataframes and panda

## Requirements:
 - pandas
 - numpy
 - openpyxl

In [2]:
import pandas as pd
import numpy as np
import openpyxl as op

## Pre-cleaning data check
 - Uploading the dataset
 - Preliminary checks of the data

In [28]:
df = pd.read_excel('2025_index_of_multiple_deprivation.xlsx',
                  sheet_name='IMD25')

# Visually checking the first and last 5 rows of data in the IMD dataset
# Including key information, such as total number of rows, column names and a sample of data
print(f'First five rows of data:\n{df.head(5)}')
print(f'\nLast five rows of data:\n{df.tail(5)}')

# Shape of dataset - to confirm the number of rows and columns
print(f'\nPre-cleaning IMD dataset shape:\n{df.shape}')

# Summary statistics pre-cleaning - notice that this only provides stats for numerical data (the IMD rank and IMD decile)
print(f'\nPre-cleaning IMD summary statistics:\n{df.describe()}')


First five rows of data:
  LSOA code (2021)           LSOA name (2021)  \
0        E01000001        City of London 001A   
1        E01000002        City of London 001B   
2        E01000003        City of London 001C   
3        E01000005        City of London 001E   
4        E01000006  Barking and Dagenham 016A   

  Local Authority District code (2024) Local Authority District name (2024)  \
0                            E09000001                       City of London   
1                            E09000001                       City of London   
2                            E09000001                       City of London   
3                            E09000001                       City of London   
4                            E09000002                 Barking and Dagenham   

   Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)  \
0                                              26525                     
1                                              31203     

## Data Cleaning

Data quality checks, including:
 - Missing values
 - Data types
 - Unrealistic/unexpected values
 - Duplicate values

In [38]:
# Copying the IMD dataset, in order to preserve the original dataset
df_copy = df.copy()

# Checking for missing values, with none found across all columns
missing_values = df_copy.isnull().sum()
print(f'Total missing values for the Index of Multiple Deprivation \
(IMD) dataset: {missing_values.sum()}')

# Investigating data types and assessing if they are the most appropriate for the values
print(f'\nAll data types for the IMD dataset: \nLSOA Code: {df_copy['LSOA code (2021)'].dtypes}\
\nLSOA Name: {df_copy['LSOA name (2021)'].dtypes}\nLAD Code: {df_copy['Local Authority District code (2024)'].dtypes}\
\nLAD Name: {df_copy['Local Authority District name (2024)'].dtypes}\nIMD Rank: \
{df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)'].dtypes}\
\nIMD Decile: {df_copy['Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA'].dtypes}')

# Checking that the IMD rank is between 1 and 33755, and that the IMD decile is between 1 and 10
if df_copy[(df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)'] < 1) | \
    (df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)']> 33755)].shape[0]:
    print(f'\nUnexpected IMD Rank values:\n{df_copy[(df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)']\
    < 1) | (df_copy['Index of Multiple Deprivation (IMD) Rank (where 1 is most deprived)']> 33755)].values}')
else:
    print('\nNo unexpected IMD Rank values.')

if df_copy[(df_copy['Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA'] < 1) | \
    (df_copy['Index of Multiple Deprivation (IMD) Decile (where 1 is most deprived 10% of LSOA']> 10)].shape[0]:
    print(f'\nUnexpected IMD Decile values:\n{df_copy[(df_copy['Index of Multiple Deprivation (IMD) \
    Decile (where 1 is most deprived 10% of LSOA'] < 1) | (df_copy['Index of Multiple Deprivation (IMD) Decile \
    (where 1 is most deprived 10% of LSOA']> 10)].values}')
else:
    print('\nNo unexpected IMD Decile values.')

Total missing values for the Index of Multiple Deprivation (IMD) dataset: 0

All data types for the IMD dataset: 
LSOA Code: str
LSOA Name: str
LAD Code: str
LAD Name: str
IMD Rank: int64
IMD Decile: int64

No unexpected IMD Rank values.

No unexpected IMD Decile values.


All data types are suitable:
 - The LSOA code, LSOA name, LAD code and LAD name columns are all strings, which is appropriate as they contain either solely letters or both numbers and letters
 - The IMD rank and IMD decile are integers, which is the most appropriate data type for these numerical values 

In [ ]:
# Remove unnecessary columns, such as the LSOA codes because unnecessary for analysis
df_imd = df.drop(['LSOA code (2021)', 'LSOA name (2021)'], axis=columns)

# Shape of dataset - to confirm the columns have been deleted successfully
print(f'IMD dataset shape:\n{df_imd.shape}')

# check for outliers with IQR method and remove outliers?

# Check for duplicates (see slides)


The LSOA name and LSOA 

## Standardising locations 

Mapping Local Authority District (LAD) codes against International Territorial Levels (ITLs), specifically ITL1 and ITL2.

I decided to  use the LAD codes, rather than the LSOA codes, to map to the ITLs because the Office for National Statistics (ONS) publish a 'Local Authority District (December 2024) to LAU1 to ITL3 to ITL2 to ITL1 (January 2025) Lookup in the UK' file. I selected this file as the LADs used in the Index of Multiple Deprivation dataset were 2024 codes, whilst the ITLs we decided to use for this project are 2025. therefore this dataset is appropriate

I decided to map LAD to ITL, rather than LOA to ITL, as this is a more simple process and one that is directly supported by the ONS. The dataset mentioned previously - - can be used to directly look up ITLs (both ITL1 and ITL2) from LADs.


The ITL1 and ITL2 regions can be utilised to *** ultiple datasets, which utilise different regions and districts, such as the police dataset and the MHCLG dataset, which utilise ** and LSOA/LAD respectively. Without 


The LAD_2024_to_ITL_2025 ONS dataset contains location data for England, Wales, Scotland and Northern Ireland, whereas the Index for Multiple Deprivation dataset only contains data for England. Therefore, I've decided to use a ** merge, to retain all location data from the Index for Multiple Deprivation dataset and not pull across any ITLs that aren't required (for example, Wales, Scotland and Northern Ireland).

In [ ]:
merge on LAD24CD
retain ITL125CD	ITL125NM	ITL225CD	ITL225NM
dont need ITL325CD	ITL325NM	LAU125CD	LAU125NM   ObjectId

In [15]:
df_itl = pd.read_csv('LAD_2024_to_ITL_2025.csv')


print(df_itl)

df_clean = 

    ITL125CD              ITL125NM ITL225CD          ITL225NM ITL325CD  \
0        TLC  North East (England)     TLC3       Tees Valley    TLC31   
1        TLC  North East (England)     TLC3       Tees Valley    TLC31   
2        TLC  North East (England)     TLC3       Tees Valley    TLC32   
3        TLC  North East (England)     TLC3       Tees Valley    TLC32   
4        TLC  North East (England)     TLC3       Tees Valley    TLC33   
..       ...                   ...      ...               ...      ...   
357      TLN      Northern Ireland     TLN0  Northern Ireland    TLN0C   
358      TLN      Northern Ireland     TLN0  Northern Ireland    TLN0D   
359      TLN      Northern Ireland     TLN0  Northern Ireland    TLN0E   
360      TLN      Northern Ireland     TLN0  Northern Ireland    TLN0F   
361      TLN      Northern Ireland     TLN0  Northern Ireland    TLN0G   

                            ITL325NM   LAU125CD                  LAU125NM  \
0    Hartlepool and Stockton-on-Te

## Renaming columns for ease of use

In [ ]:
# rename columns .str.replace('','')



## Clean dataset

In [ ]:
# Shape of dataset - post-cleaning - to confirm the number of rows and columns
print(f'\nPost-cleaning IMD dataset shape:\n{df_clean.shape}')

# Summary statistics post-cleaning - notice that this only provides stats for numerical data (the IMD rank and IMD decile)
print(f'\nPost-cleaning summary statistics:\n{df_clean.describe()}')

# Save the clean dataset as an Excel file
df_clean.to_excel('', index=False)